In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json

import os
import numpy as np

import warnings
warnings.filterwarnings('ignore')

import MEArec as mr
import pandas as pd

In [2]:
recording, sorting = se.read_mearec("/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5")

In [3]:
recording_recorded = spre.bandpass_filter(recording, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

In [ ]:
output_folder = '/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s'
recording_preprocessed = recording_f.save(format="binary", n_jobs = 20)

default_params = {
        'detect_sign': -1,  
        'adjacency_radius': 120, 
        'freq_min': 300,  
        'freq_max': 3000,
        'filter': True,
        'whiten': True,  
        'num_workers': 20,
        'clip_size': 50,
        'detect_threshold': 4,
        'detect_interval': 3,  
    }
sorting_mountainsort = ss.run_sorter(sorter_name='mountainsort4',
                                recording=recording_preprocessed,
                                remove_existing_folder='True',
                                folder=output_folder,
                                **default_params)

analyzer_mountainsort = si.create_sorting_analyzer(
    sorting=sorting_mountainsort, 
    recording=recording_preprocessed, 
    format='binary_folder', 
    folder=output_folder + '/analyzer_kilosort4_binary'
)

# 计算扩展信息
extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 20)

# 读取spikes.npy并检查无效的spike
spikes_path = output_folder + "/analyzer_kilosort4_binary/sorting/spikes.npy"
spikes = np.load(spikes_path)

# 获取recording的总样本数
total_samples = recording_f.get_num_samples()

# 检查第一个和最后一个spike
first_spike_valid = spikes[0]['sample_index'] >= 0
last_spike_valid = spikes[-1]['sample_index'] < total_samples

# 如果第一个或最后一个spike无效，删除所有无效的spike
if not first_spike_valid or not last_spike_valid:
    # 创建有效spike的掩码：sample_index >= 0 且 < total_samples
    valid_mask = (spikes['sample_index'] >= 0) & (spikes['sample_index'] < total_samples)
    spikes_filtered = spikes[valid_mask]
    
    # 保存过滤后的spikes
    np.save(spikes_path, spikes_filtered)
    print(f"删除了 {len(spikes) - len(spikes_filtered)} 个无效的spike")
    print(f"原始spike数量: {len(spikes)}, 过滤后: {len(spikes_filtered)}")
else:
    print("所有spike都在有效范围内")

qm_params = sqm.get_default_qm_params()
analyzer_mountainsort.compute("quality_metrics", qm_params, n_jobs = 20)

# 导出到phy格式


Use cache_folder=/tmp/spikeinterface_cache/tmp00wbm0uo/Z6Q2NI2O
write_binary_recording 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=14.65 MiB - total_memory=292.97 MiB - chunk_duration=1.00s


write_binary_recording (workers: 20 processes):   0%|          | 0/3600 [00:00<?, ?it/s]

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording


sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:1: DeprecationWarning: Call to deprecated function (or staticmethod) _destroy.
sys:

estimate_sparsity (no parallelization):   0%|          | 0/3600 [00:00<?, ?it/s]

...

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/sortinganalyzer.py:2656: DeprecationWarning: The method 'cosine_similarity' is deprecated and will be removed in the next version. Use 'cosine' instead.
  params = self._set_params(**params)


Compute : spike_locations (workers: 20 processes):   0%|          | 0/3600 [00:00<?, ?it/s]

所有spike都在有效范围内


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1531: DeprecationWarning: `row_stack` alias is deprecated. Use `np.vstack` directly.
  inds_confidence90 = np.row_stack(np.where(conf_matrix[:, test_rp_centers_mask] > 0.9))
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1066: UserWarning: Some units have too few spikes : amplitude_cutoff is set to NaN
  warnings.warn(f"Some units have too few spikes : amplitude_cutoff is set to NaN")
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:914: UserWarning: compute_amplitude_cv_metrics() need 'spike_amplitudes' or 'amplitude_scalings'
  warnings.warn("compute_amplitude_cv_metrics() need 'spike_amplitudes' or 'amplitude_scalings'")
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1658: Us

NameError: name 'sexp' is not defined

In [5]:
import spikeinterface.exporters as sexp
sexp.export_to_phy(analyzer_mountainsort, output_folder + "/phy_folder_for_kilosort", verbose=True, n_jobs = 20)

write_binary_recording (workers: 20 processes):   0%|          | 0/3600 [00:00<?, ?it/s]

spike_amplitudes (workers: 20 processes):   0%|          | 0/3600 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/981 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/981 [00:00<?, ?it/s]

extract PCs (workers: 20 processes):   0%|          | 0/3600 [00:00<?, ?it/s]

Run:
phy template-gui  /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/phy_folder_for_kilosort/params.py
